# First look at the data

Exploration only, not pipeline code. Goals:
1. Download 5 years of **raw** AAPL prices (`auto_adjust=False`) and look at their shape.
2. Look at AAPL's splits and dividends.
3. Save the S&P 500 ticker list to `data/reference/sp500_tickers.csv`.

In [1]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

print("yfinance", yf.__version__, "| pandas", pd.__version__)

yfinance 1.8.0 | pandas 3.1.1


## 1. Daily prices: AAPL, last 5 years

In [2]:
# auto_adjust=False keeps raw Close and gives a separate 'Adj Close' column.
prices = yf.download("AAPL", period="5y", auto_adjust=False, progress=False)
print("shape:", prices.shape)
print("index:", prices.index.dtype, "| from", prices.index.min().date(), "to", prices.index.max().date())
print("columns:", list(prices.columns))
prices.tail()

shape: (1255, 6)
index: datetime64[s] | from 2021-11-15 to 2026-11-13
columns: [('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2026-11-09,350.423737,350.690002,352.329987,344.779999,345.000000,48753100
2026-11-10,348.245392,348.510010,353.869995,346.459991,351.350006,29756100
2026-11-11,355.359985,355.630005,355.910004,349.329987,349.549988,34453500
2026-11-12,354.880005,354.880005,359.369995,354.410004,357.809998,31875000
2026-11-13,356.269989,356.269989,360.829987,355.750000,357.959991,39985300


In [3]:
# Column header is a 2-level MultiIndex (Price, Ticker), even for one ticker.
# Flatten it so the frame is easier to read.
aapl = prices.xs("AAPL", axis=1, level="Ticker")
aapl.dtypes

Price
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
dtype: object

In [4]:
# Basic quality checks we'll later write as dbt tests.
print("rows:", len(aapl))
print("nulls per column:", aapl.isna().sum().to_dict())
print("duplicate dates:", aapl.index.duplicated().sum())
print("high >= low everywhere:", (aapl["High"] >= aapl["Low"]).all())
print("all prices > 0:", (aapl[["Open", "High", "Low", "Close", "Adj Close"]] > 0).all().all())
aapl.describe().round(2)

rows: 1255
nulls per column: {'Adj Close': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
duplicate dates: 0
high >= low everywhere: True
all prices > 0: True


Price,Adj Close,Close,High,Low,Open,Volume
count,1255.00,1255.00,1255.00,1255.00,1255.00,1.255000e+03
mean,207.50,209.54,211.65,207.24,209.30,6.291336e+07
std,54.89,54.02,54.45,53.53,53.93,2.865717e+07
min,122.73,125.02,127.77,124.17,126.01,1.791060e+07
25%,165.97,168.88,170.63,167.63,169.06,4.415520e+07
50%,191.69,194.03,195.64,192.55,194.10,5.428830e+07
75%,242.86,244.54,246.15,241.86,244.23,7.482350e+07
max,356.27,356.27,360.83,355.75,357.96,3.186799e+08


In [5]:
# Close vs Adj Close: how big is the dividend adjustment?
ratio = (aapl["Adj Close"] / aapl["Close"]).rename("adj_ratio")
pd.concat([ratio.head(3), ratio.tail(3)])

Date
2021-11-15    0.976053
2021-11-16    0.976053
2021-11-17    0.976053
2026-11-11    0.999241
2026-11-12    1.000000
2026-11-13    1.000000
Name: adj_ratio, dtype: float64

## 2. Splits and dividends

In [6]:
ticker = yf.Ticker("AAPL")
splits = ticker.splits
print("index dtype:", splits.index.dtype)
splits

index dtype: datetime64[us, America/New_York]


Date
1987-06-16 09:30:00-04:00    2.0
2000-06-21 09:30:00-04:00    2.0
2005-02-28 09:30:00-05:00    2.0
2014-06-09 09:30:00-04:00    7.0
2020-08-31 09:30:00-04:00    4.0
Name: Stock Splits, dtype: float64

In [7]:
dividends = ticker.dividends
print("dividend rows (full history):", len(dividends))
dividends.tail(8)

dividend rows (full history): 93


Date
2025-02-10 09:30:00-05:00    0.25
2025-05-12 09:30:00-04:00    0.26
2025-08-11 09:30:00-04:00    0.26
2025-11-10 09:30:00-05:00    0.26
2026-02-09 09:30:00-05:00    0.26
2026-05-11 09:30:00-04:00    0.27
2026-08-10 09:30:00-04:00    0.27
2026-11-12 09:30:00-05:00    0.27
Name: Dividends, dtype: float64

### Is `Close` really raw? Check the 2020-08-31 4:1 split

The 5-year window starts after AAPL's last split, so it doesn't show what a split does to the prices.
Pull the days around the August 2020 split. On the real tape, AAPL closed at about **$499** on 2020-08-28 and about **$129** on 2020-08-31.

In [8]:
around_split = yf.download("AAPL", start="2020-08-25", end="2020-09-03", auto_adjust=False, progress=False)
around_split.xs("AAPL", axis=1, level="Ticker")[["Close", "Adj Close", "Volume"]]

Price,Close,Adj Close,Volume
Date,,,
2020-08-25,124.824997,120.880791,211495600
2020-08-26,126.522499,122.524651,163022400
2020-08-27,125.010002,121.059937,155552400
2020-08-28,124.807503,120.863838,187630000
2020-08-31,129.039993,124.962570,225702700
2020-09-01,134.179993,129.940170,151948100
2020-09-02,131.399994,127.248039,200119000


## 3. S&P 500 ticker list from Wikipedia

In [9]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
# Wikipedia returns 403 to Python's default User-Agent, so send our own.
headers = {"User-Agent": "stock-market-elt-pipeline/0.1 (portfolio project)"}
resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

# Table 0 = current constituents. Read CIK as a string so the leading zeros
# survive (SEC EDGAR uses 10-digit zero-padded CIKs).
sp500 = pd.read_html(StringIO(resp.text), converters={"CIK": str})[0]
print("rows:", len(sp500), "| unique symbols:", sp500["Symbol"].nunique())
print("columns:", list(sp500.columns))
sp500.head()

rows: 503 | unique symbols: 503
columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [10]:
# Symbols with a dot: Yahoo uses a dash instead (BRK.B -> BRK-B).
sp500[sp500["Symbol"].str.contains(r"\.")][["Symbol", "Security"]]

,Symbol,Security
60,BRK.B,Berkshire Hathaway
76,BF.B,Brown–Forman


In [11]:
out = Path("../data/reference/sp500_tickers.csv")
sp500.to_csv(out, index=False)
print("saved", len(sp500), "rows to", out.resolve())

saved 503 rows to /workspaces/stock-market-elt-pipeline/data/reference/sp500_tickers.csv
